In [235]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [236]:
#pip install tensorflow

In [237]:
# Here we have a small dataset of texts with positive, negative, and neutral sentiments. We will use this dataset to train a simple sentiment analysis model using a transformer architecture. 
# if we removet this part, we will not have any data to train our model on.
texts = [
    # Positive
    "I love this movie",
    "Excellent story",
    "I love chicken",
    "This is amazing and wonderful",
    "Best experience ever",
    "Absolutely fantastic work",
 
    # Negative
    "I hate this movie",
    "Mcdonalds is bad",
    "Terrible service and rude staff",
    "This is the worst thing ever",
    "Awful experience never again",
    "I really dislike this product",
 
    # Neutral
    ##"It's a really good movie but very lengthy",
    #"Loved the storyline but not the cinematography.",
    #"Superb product but not really healthy",
    #"Loved this a lot but not much."
]

In [238]:
# here we add labels for the sentiments. 1 for positive, 0 for negative.
# if we remove this part we will get error during training because the model will not know what to learn from the data
labels = np.array([1, 1, 1, 1, 1, 1,
                   0, 0, 0, 0, 0, 0])

In [239]:
vocab_size = 1000
max_length = 6
tokenizer = Tokenizer(num_words = vocab_size, oov_token ="<OOV>")
#Out Of Vocablary (OOV) token is used to represent words that are not in the vocabulary. It helps the model handle unseen words during inference.
# if we skip the topenization step we cannot convert text sequence diretly in to integers and we will get error during training because the model expects numerical input.
tokenizer.fit_on_texts(texts)

# we tokeinze the texts and convert them into sequences of integers. Each integer corresponds to a word in the vocabulary. 
sequences = tokenizer.texts_to_sequences(texts)

# here we also pad the sequences to ensure they all have the same length, which is necessary for training the model.
# if we dont pad there will be shape mismatch error during training because the model expects input of a fixed size.
X = pad_sequences(sequences, maxlen=max_length, padding='post')
print("Word Index:", tokenizer.word_index)
print("Padded Sequences(Input Sequences):\n", X)


Word Index: {'<OOV>': 1, 'this': 2, 'i': 3, 'is': 4, 'love': 5, 'movie': 6, 'and': 7, 'experience': 8, 'ever': 9, 'excellent': 10, 'story': 11, 'chicken': 12, 'amazing': 13, 'wonderful': 14, 'best': 15, 'absolutely': 16, 'fantastic': 17, 'work': 18, 'hate': 19, 'mcdonalds': 20, 'bad': 21, 'terrible': 22, 'service': 23, 'rude': 24, 'staff': 25, 'the': 26, 'worst': 27, 'thing': 28, 'awful': 29, 'never': 30, 'again': 31, 'really': 32, 'dislike': 33, 'product': 34}
Padded Sequences(Input Sequences):
 [[ 3  5  2  6  0  0]
 [10 11  0  0  0  0]
 [ 3  5 12  0  0  0]
 [ 2  4 13  7 14  0]
 [15  8  9  0  0  0]
 [16 17 18  0  0  0]
 [ 3 19  2  6  0  0]
 [20  4 21  0  0  0]
 [22 23  7 24 25  0]
 [ 2  4 26 27 28  9]
 [29  8 30 31  0  0]
 [ 3 32 33  2 34  0]]


In [240]:
# token and position embedding
# in this block we define a custom layer that combines token embeddings and positional embeddings. The token embedding converts each word (represented as an integer) into a dense vector of fixed size (embed_dim). 
# The positional embedding adds information about the position of each word in the sequence, which is crucial for the transformer to understand the order of words. The call method computes the token and position embeddings and returns their sum, which will be used as input to the transformer layers.
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        # Token embedding
        # if we remove the token embedding layer, the model will not be able to learn meaningful representations of the words in the input sequences, which will lead to poor performance during training and inference.
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
           output_dim=embed_dim
        )

        #super().__init__()
        #position embedding
        # if we remove the position embedding layer, the model will lose the ability to capture the order of words in the input sequences, which is crucial for understanding the context and meaning of the text. This will lead to poor performance during training and inference, especially for tasks that require understanding of word relationships and sequence structure.
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb
        #return position_emb

In [ ]:
# in the block below we define a transformer block, which consists of a multi-head self-attention layer followed by a feed-forward neural network. The self-attention layer allows the model to weigh the importance of different words in the input sequence when making predictions, while the feed-forward network helps to capture complex relationships between the features extracted by the attention mechanism. The layer normalization and skip connections help to stabilize training and improve performance.
# if we remove the transformer block, the model will not be able to capture the complex relationships and dependencies between words in the input sequence, eg it will not be able to understand that "I love this movie" is positive and "I hate this movie" is negative because it will not be able to capture the relationship between "love" and "hate" with "movie".
def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        # wht the following acticvatiosn does 
        # softmax : if we use it here, it will convert the attention scores into probabilities, which can help the model focus on the most relevant parts of the input sequence. 
        # relu : if we use it here, it will introduce non-linearity into the feed-forward network, allowing the model to learn more complex patterns and relationships in the data. 
        # sigmoid : it causes the output of the feed-forward network to be between 0 and 1, which can be useful for certain types of tasks, such as binary classification.but here wee have complex patterns in the data
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        # if we remove the attention layer, the model will not be able to capture the dependencies and relationships between different words in the input sequence. 
        
        #multihead attention
        # Different attention heads learn different patterns.
        # Captures context.If removed model behaves much more simply.misses important relationships.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        # if we  remove this part, the model may suffer from issues like unstable training, which can lead topoor performance and convergence problems.
        # normalizations Keeps activations stable it is needed coz it Prevents exploding/unstable values.if removed Training may become unstable.Loss may oscillate.
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        # this part processes attention outputs and learns higher-level features.
        # Attention alone is not enough.If removed Model becomes much weak
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2


In [ ]:
# build the model

#embed_dim = 8
#num_heads = 1
# time fit = 6.1 secs
# accuracy 1.0 at 5ms/step, 6th epoch
# Epoch 8/30
# 6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.5110 


#embed_dim = 16
#num_heads = 2
# time fit = 7.1 secs, 5ht epoch - acc =1 (8ms/step)
# Epoch 9/30
# 6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.2562

#embed_dim = 32
#num_heads = 4
# time fit =6th epoch - acc =1 (6ms/step)
#Epoch 6/30
# Epoch 3/30
# 6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.4955 

embed_dim = 64
num_heads = 8
#Epoch 5/30
#6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.2677
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
# if we remove the transformer block, the model will not be able to capture the complex relationships and dependencies between words in the input sequence, which is crucial for understanding the context and meaning of the text. 
# for eg it will not be able to understand that "I love this movie" is positive and "I hate this movie" is negative because it will not be able to capture the relationship between "love" and "hate" with "movie". 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
#output layer
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [243]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_38 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_25 │ (None, 6, 64)          │        64,384 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_21            │ (None, 6, 64)          │       137,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_16     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_60 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,569 (787.38 KB)

 Trainable params: 201,569 (787.38 KB)

 Non-trainable params: 0 (0.00 B)

In [244]:
model.fit(X, labels, epochs=30, batch_size=2, verbose=1)

Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.4167 - loss: 1.1330   
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6667 - loss: 0.5787
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7500 - loss: 0.5286 
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8333 - loss: 0.3688 
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.2677
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1935 
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.1510 
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0542 
Epoch 9/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0738 
Epoch 10/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0280 
Epoch 11/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0198 
Epoch 12/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0075

In [245]:
test_sentences = [
    "I love the film",
    "This movie was awful"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
I love the film -> 0.9375998
Prediction: Positive
This movie was awful -> 0.2552075
Prediction: Negative


TASK 2 : DATA FLOW

Input data
    |
Add Labels
    |
Tokenize
    |
Padding
    |
Token Embedding
    |
Positional Embedding
    |
Multihead Attention
    |
add + Normalize
    |
Feedforward neural network
    |
add + normalize

